# 02 - Hyperspectral Data Analysis

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook covers hyperspectral data cube handling, spectral signature extraction, and analysis using:
- **EnMAP** (30m, 205 bands, 400-2500nm)
- **Planet Tanager** (20m, 205 bands, 400-1700nm)
- **HyperCoast** library for interactive visualization
- Custom spectral analysis for soil salinity, vegetation stress, mineralogy

**Target:** Liwa Oasis agricultural fields - soil salinity mapping, crop health assessment

In [ ]:
# Install required packages
!pip -q install hypercoast spectral rioxarray xarray matplotlib numpy pandas folium

In [ ]:
import hypercoast
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray
from pathlib import Path

print(f"HyperCoast version: {hypercoast.__version__}")
print("Available readers:")
for reader in ['read_enmap', 'read_tanager', 'read_emit', 'read_prisma', 'read_aviris', 'read_des']:
    print(f"  {reader}")

## 1. Reading Hyperspectral Data Cubes

In [ ]:
# Template for reading EnMAP data
# EnMAP data must be downloaded from DLR geoservice first
# Structure: ENMAP01-_____HSI______T_____V010001_20240101T000000Z/

def read_enmap_cube(enmap_dir):
    """Read EnMAP hyperspectral cube from directory."""
    # EnMAP delivers as multiple files - use HyperCoast reader
    # ds = hypercoast.read_enmap(enmap_dir)
    # return ds
    pass

# Template for reading Planet Tanager data
# Tanager open archive delivers COGs via STAC

def read_tanager_cube(item):
    """Read Planet Tanager cube from STAC item."""
    # ds = hypercoast.read_tanager(item.assets['radiance'].href)
    # return ds
    pass

print("Reader templates defined. Provide actual data paths to use.")

## 2. Spectral Signature Extraction & Analysis

In [ ]:
def extract_spectral_signatures(cube, wavelengths, coordinates, labels=None):
    """
    Extract spectral signatures from hyperspectral cube at given coordinates.
    
    Parameters:
    - cube: xarray DataArray (y, x, band) or (band, y, x)
    - wavelengths: array of wavelength values (nm)
    - coordinates: list of (y, x) pixel coordinates
    - labels: optional labels for each coordinate
    
    Returns:
    - signatures: array of shape (n_coords, n_bands)
    - wavelengths: array of wavelength values
    """
    if cube.dims[0] == 'band':
        cube = cube.transpose('y', 'x', 'band')
    
    signatures = []
    for y, x in coordinates:
        spec = cube.isel(y=y, x=x).values
        signatures.append(spec)
    
    signatures = np.array(signatures)
    
    if labels is None:
        labels = [f"Pixel ({y},{x})" for y, x in coordinates]
    
    return signatures, wavelengths

In [ ]:
def plot_spectral_signatures(signatures, wavelengths, labels, title="Spectral Signatures"):
    """Plot multiple spectral signatures."""
    plt.figure(figsize=(12, 6))
    for i, (spec, label) in enumerate(zip(signatures, labels)):
        plt.plot(wavelengths, spec, label=label, linewidth=1.5)
    
    plt.axvspan(400, 700, alpha=0.1, color='blue', label='Visible')
    plt.axvspan(700, 1300, alpha=0.1, color='green', label='NIR')
    plt.axvspan(1300, 2500, alpha=0.1, color='red', label='SWIR')
    
    plt.xlabel('Wavelength (nm)')
    plt.ylabel('Reflectance / Radiance')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 3. Key Hyperspectral Indices for Agriculture

In [ ]:
def compute_hyperspectral_indices(cube, wavelengths):
    """
    Compute key hyperspectral indices for agriculture and soil analysis.
    
    Indices:
    - Chlorophyll: MCARI, TCARI, CIred_edge
    - Water: WI, NDWI (hyperspectral versions)
    - Soil/Salinity: SI, BI, SI1, SI2, SI3
    - Nitrogen: NNI
    - Carotenoids: CARI, SIPI
    """
    def find_band(target_nm):
        return np.argmin(np.abs(wavelengths - target_nm))
    
    b670 = find_band(670)
    b700 = find_band(700)
    b750 = find_band(750)
    b800 = find_band(800)
    b970 = find_band(970)
    b1200 = find_band(1200)
    b1600 = find_band(1600)
    b2100 = find_band(2100)
    b2200 = find_band(2200)
    
    if cube.dims[0] == 'band':
        cube = cube.transpose('y', 'x', 'band')
    
    indices = {}
    
    indices['MCARI'] = ((cube[..., b750] - cube[..., b670]) - 0.2 * (cube[..., b750] - cube[..., b700])) * (cube[..., b750] / cube[..., b670])
    indices['TCARI'] = 3 * ((cube[..., b700] - cube[..., b670]) - 0.2 * (cube[..., b700] - cube[..., b550]) * (cube[..., b700] / cube[..., b670]))
    
    indices['WI_970'] = cube[..., b970] / cube[..., b900]
    indices['NDWI_HS'] = (cube[..., b860] - cube[..., b1240]) / (cube[..., b860] + cube[..., b1240])
    
    indices['SI'] = (cube[..., b1600] - cube[..., b670]) / (cube[..., b1600] + cube[..., b670])
    indices['BI'] = np.sqrt(cube[..., b1600]**2 + cube[..., b670]**2)
    indices['SI1'] = cube[..., b1600] / cube[..., b800]
    indices['SI3'] = (cube[..., b2200] - cube[..., b1600]) / (cube[..., b2200] + cube[..., b1600])
    
    return indices

## 4. HyperCoast Interactive Visualization

In [ ]:
# HyperCoast provides interactive map with spectral signature plotting
# Usage example (requires actual data):

# import hypercoast
# 
# # For EnMAP
# ds = hypercoast.read_enmap("path/to/enmap/data")
# m = hypercoast.Map()
# m.add_enmap(ds, vmin=0, vmax=0.3)
# m.add("spectral")  # Click any pixel to plot its spectral signature
# m

# For Tanager
# ds = hypercoast.read_tanager("path/to/tanager/data")
# m = hypercoast.Map()
# m.add_tanager(ds, vmin=0, vmax=0.3)
# m.add("spectral")
# m

print("HyperCoast visualization ready. Provide data path to use.")

## 5. Spectral Unmixing & Endmember Extraction

In [ ]:
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler

def spectral_unmixing(cube, n_endmembers=4):
    """
    Perform spectral unmixing using Non-negative Matrix Factorization.
    
    Parameters:
    - cube: xarray DataArray (y, x, band)
    - n_endmembers: number of pure spectral endmembers to extract
    
    Returns:
    - abundances: (y, x, n_endmembers) fractional abundances
    - endmembers: (n_endmembers, n_bands) pure spectral signatures
    """
    if cube.dims[0] == 'band':
        cube = cube.transpose('y', 'x', 'band')
    
    h, w, b = cube.shape
    pixels = cube.values.reshape(-1, b)
    
    valid_mask = np.all(np.isfinite(pixels), axis=1)
    pixels_valid = pixels[valid_mask]
    
    scaler = MinMaxScaler()
    pixels_scaled = scaler.fit_transform(pixels_valid)
    
    nmf = NMF(n_components=n_endmembers, random_state=42, max_iter=500)
    abundances_scaled = nmf.fit_transform(pixels_scaled)
    endmembers_scaled = nmf.components_
    
    abundances_full = np.zeros((h * w, n_endmembers))
    abundances_full[valid_mask] = abundances_scaled
    abundances = abundances_full.reshape(h, w, n_endmembers)
    
    endmembers = scaler.inverse_transform(endmembers_scaled)
    
    return abundances, endmembers

## 6. Next Steps

In [ ]:
print("Hyperspectral Analysis Pipeline Ready")
print("=" * 50)
print("Next steps for Liwa AOI:")
print("1. Download EnMAP scene covering Liwa (DLR geoservice)")
print("2. Search Planet Tanager open archive for Liwa coverage")
print("3. Run spectral signature extraction on field boundaries")
print("4. Compute salinity indices (SI, BI, SI3) for soil mapping")
print("5. Compute chlorophyll/water indices for crop health")
print("6. Perform spectral unmixing to identify surface materials")
print("7. Validate with ground truth (if available)")